In [59]:
import numpy as np
import json
import os
import re
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 8)

In [60]:
def parse_experiment_folder(folder_name):
    """
    Parse ablation_results folder name.
    Format: {date}_{time}_{model}_attn{True|False}_lam{lambda}_m{margin}_{hash}
    Example: 20260518_054245_bert-base-uncased_attnTrue_lam0.3_m0.1_85494943
    """
    pattern = r'^\d+_\d+_(.+?)_attn(True|False)_lam([\d.]+)_m([\d.]+)_\w+$'
    match = re.match(pattern, folder_name)
    if match:
        model_name = match.group(1)
        attn = match.group(2)
        lam = match.group(3)
        margin = match.group(4)
        return model_name, attn, lam, margin
    return None, None, None, None


def make_config_name(attn, lam, margin):
    """Derive a readable config name from hyperparameter values."""
    if attn == 'False':
        return 'baseline'
    # Paper config (anchor point)
    if lam == '0.3' and margin == '0.1':
        return 'haxe'
    # Lambda sweep (margin fixed at 0.1)
    if margin == '0.1':
        return f'lam{lam}'
    # Margin sweep (lambda fixed at 0.3)
    if lam == '0.3':
        return f'm{margin}'
    # Fallback
    return f'lam{lam}_m{margin}'


def discover_experiments(base_path='ablation_results'):
    """Discover all experiments in ablation_results folder."""
    experiments = defaultdict(dict)

    base = Path(base_path)
    if not base.exists():
        print(f"Error: {base_path} folder not found!")
        return experiments

    for folder in base.iterdir():
        if not folder.is_dir():
            continue

        model_name, attn, lam, margin = parse_experiment_folder(folder.name)
        if model_name is None:
            continue

        model_key = model_name.split('-')[0]  # bert-base-uncased -> bert
        config_name = make_config_name(attn, lam, margin)

        results_file = folder / 'results' / 'test_explain_output.jsonl'
        if results_file.exists():
            experiments[model_key][config_name] = {
                'path': str(results_file),
                'folder': str(folder),
                'full_model_name': model_name,
                'lambda': lam,
                'margin': margin,
                'attn': attn,
            }

    return experiments


def load_jsonl(filepath):
    """Load JSONL file."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data


# ── Discover experiments ────────────────────────────────────────────────────
print("=" * 80)
print("DISCOVERING EXPERIMENTS IN ablation_results FOLDER")
print("=" * 80)

experiments = discover_experiments(base_path='ablation_results')

print(f"\nFound {len(experiments)} model(s):")
for model_key in sorted(experiments.keys()):
    configs = sorted(experiments[model_key].keys())
    print(f"  {model_key.upper()}: {len(configs)} configurations -> {configs}")

# ── Load ground truth ───────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("LOADING GROUND TRUTH DATA")
print("=" * 80)

GT_PATH = Path('Data/bert_ground_truth/test.jsonl')
gt_model = {}

if GT_PATH.exists():
    print(f"\nLoading ground truth from {GT_PATH}...")
    json_truth = load_jsonl(GT_PATH)
    gt_model['bert'] = {}
    for item in json_truth:
        annotation_id = item['annotation_id']
        gt_model['bert'][annotation_id] = {
            'evidences': item.get('evidences', []),
            'classification': item.get('classification', '0'),
        }
    print(f"  ✓ Loaded {len(gt_model['bert'])} ground truth annotations")
else:
    print(f"  ✗ Ground truth not found at {GT_PATH}")

# ── Load model predictions ──────────────────────────────────────────────────
print("\n" + "=" * 80)
print("LOADING MODEL PREDICTIONS")
print("=" * 80)

model_data = {}
for model_key, configs in experiments.items():
    print(f"\nLoading {model_key.upper()} predictions...")
    model_data[model_key] = {}
    for config_name, config_info in sorted(configs.items()):
        data = load_jsonl(config_info['path'])
        model_data[model_key][config_name] = data
        print(f"  {config_name:>12}: {len(data)} samples  "
              f"(λ={config_info['lambda']}, m={config_info['margin']})")

print("\n" + "=" * 80)
print("✓ ALL DATA LOADED SUCCESSFULLY!")
print("=" * 80)


DISCOVERING EXPERIMENTS IN ablation_results FOLDER

Found 1 model(s):
  BERT: 11 configurations -> ['baseline', 'haxe', 'lam0.1', 'lam0.5', 'lam1.0', 'lam100.0_m100.0', 'lam2.0', 'm0.05', 'm0.2', 'm0.3', 'm0.5']

LOADING GROUND TRUTH DATA

Loading ground truth from Data\bert_ground_truth\test.jsonl...
  ✓ Loaded 1142 ground truth annotations

LOADING MODEL PREDICTIONS

Loading BERT predictions...
      baseline: 1142 samples  (λ=0.0, m=0.1)
          haxe: 1142 samples  (λ=0.3, m=0.1)
        lam0.1: 1142 samples  (λ=0.1, m=0.1)
        lam0.5: 1142 samples  (λ=0.5, m=0.1)
        lam1.0: 1142 samples  (λ=1.0, m=0.1)
  lam100.0_m100.0: 1142 samples  (λ=100.0, m=100.0)
        lam2.0: 1142 samples  (λ=2.0, m=0.1)
         m0.05: 1142 samples  (λ=0.3, m=0.05)
          m0.2: 1142 samples  (λ=0.3, m=0.2)
          m0.3: 1142 samples  (λ=0.3, m=0.3)
          m0.5: 1142 samples  (λ=0.3, m=0.5)

✓ ALL DATA LOADED SUCCESSFULLY!


In [61]:
def create_ground_truth_mask(evidences, num_tokens):
    """Create binary mask for ground truth rationales"""
    mask = np.zeros(num_tokens)
    
    if not evidences or not evidences[0]:  # Empty evidences
        return mask
    
    for evidence_list in evidences:
        for evidence in evidence_list:
            start = evidence.get('start_token', -1)
            end = evidence.get('end_token', -1)
            
            if start >= 0 and end >= 0 and start < num_tokens and end <= num_tokens:
                mask[start:end] = 1
    
    return mask

In [68]:
def calculate_attention_entropy(attention_scores):
    """Calculate entropy of attention distribution (lower = more focused)"""
    att_scores = np.array(attention_scores, dtype=np.float64)
    # Calculate entropy
    entropy = -np.sum(att_scores * np.log(att_scores + 1e-10))
    return entropy

# Calculate entropy for all models and configurations
print("=" * 80)
print("CALCULATING ENTROPY FOR ALL CONFIGURATIONS")
print("=" * 80)

for model_key in sorted(model_data.keys()):
    print(f"\n{model_key.upper()} Entropy Results:")
    print("-" * 80)
    
    # First pass: calculate all entropies
    config_stats = {}
    # Create a list of config names to avoid modifying dict during iteration
    config_names = [k for k in model_data[model_key].keys() if not k.endswith('_entropies')]
    
    for config_name in config_names:
        entropies = []
        for sample in model_data[model_key][config_name]:
            rationales = sample['rationales'][0]
            attention_scores = rationales['soft_rationale_predictions']
            entropy = calculate_attention_entropy(attention_scores)
            entropies.append(entropy)
        
        # Store entropies in model_data
        model_data[model_key][f'{config_name}_entropies'] = entropies
        
        # Store stats for sorting
        config_stats[config_name] = {
            'mean': np.mean(entropies),
            'std': np.std(entropies)
        }
    
    # Sort by mean entropy (highest to lowest)
    sorted_configs = sorted(config_stats.items(), key=lambda x: x[1]['mean'], reverse=True)
    
    # Get highest entropy for percentage calculation
    highest_entropy = sorted_configs[0][1]['mean']
    
    # Display sorted results with ranking
    print(f"{'Rank':<6} {'Config':<12} {'Avg Entropy':<12} {'Std':<10} {'Change %':<10}")
    print("-" * 80)
    
    for rank, (config_name, stats) in enumerate(sorted_configs, 1):
        avg_entropy = stats['mean']
        std_entropy = stats['std']
        
        # Calculate percentage change from highest
        if rank == 1:
            change_pct = 0.0
            change_str = "—"
        else:
            change_pct = ((avg_entropy - highest_entropy) / highest_entropy) * 100
            change_str = f"{change_pct:+.2f}%"
        
        print(f"#{rank:<5} {config_name:<12} {avg_entropy:<12.4f} {std_entropy:<10.4f} {change_str:<10}")

print("\n" + "=" * 80)
print("✓ ENTROPY CALCULATION COMPLETE!")
print("=" * 80)

CALCULATING ENTROPY FOR ALL CONFIGURATIONS

BERT Entropy Results:
--------------------------------------------------------------------------------
Rank   Config       Avg Entropy  Std        Change %  
--------------------------------------------------------------------------------
#1     baseline     2.9775       0.5434     —         
#2     lam100.0_m100.0 2.8670       0.4976     -3.71%    
#3     m0.05        2.6906       0.5146     -9.64%    
#4     lam0.1       2.6102       0.5094     -12.34%   
#5     haxe         2.3158       0.4877     -22.22%   
#6     lam0.5       2.2727       0.4848     -23.67%   
#7     lam1.0       2.2459       0.4894     -24.57%   
#8     lam2.0       2.1963       0.4709     -26.24%   
#9     m0.2         2.0180       0.4947     -32.23%   
#10    m0.3         1.7794       0.5217     -40.24%   
#11    m0.5         1.3502       0.4848     -54.65%   

✓ ENTROPY CALCULATION COMPLETE!


In [63]:
def visualize_attention_example(annotation_id, model_key='bert'):
    """Visualize attention distribution comparison with entropy for all configurations"""
    
    # Get all configurations for this model
    configs = sorted([k for k in model_data[model_key].keys() if not k.endswith('_entropies')])
    print(f"Configs: {configs}")
    # Collect data for all configurations
    config_data = {}
    for config_name in configs:
        for item in model_data[model_key][config_name]:
            if item['annotation_id'] == annotation_id:
                config_data[config_name] = item
                break
    
    # Check if we found data for all configs
    if len(config_data) != len(configs):
        missing = set(configs) - set(config_data.keys())
        print(f"Annotation {annotation_id} not found in configurations: {missing}")
        return
    
    # Get attention scores for all configs
    attention_data = {}
    entropy_data = {}
    for config_name in configs:
        attention = config_data[config_name]['rationales'][0]['soft_rationale_predictions']
        attention_data[config_name] = attention
        entropy_data[config_name] = calculate_attention_entropy(attention)
    
    # Get ground truth
    evidences = gt_model.get(model_key, {}).get(annotation_id, {}).get('evidences', [])
    gt_mask = create_ground_truth_mask(evidences, len(list(attention_data.values())[0]))
    
    # Load document text
    doc_file = f"full/Data/explanations/docs/{annotation_id}"
    try:
        with open(doc_file, 'r', encoding='utf-8') as f:
            doc_text = f.read().strip()
    except:
        doc_text = None
    
    # Dynamic colors
    colors = ['#FF8C00', '#FFB347', '#90EE90', '#32CD32', '#1E90FF', '#9370DB', '#FF69B4']
    
    # Create subplots: one for each config + ground truth
    num_plots = len(configs) + 1
    fig, axes = plt.subplots(num_plots, 1, figsize=(18, 4 * num_plots))
    if num_plots == 1:
        axes = [axes]
    
    token_indices = np.arange(len(list(attention_data.values())[0]))
    
    # Plot each configuration
    for idx, config_name in enumerate(configs):
        attention = attention_data[config_name]
        entropy = entropy_data[config_name]
        color = colors[idx % len(colors)]
        
        # Format config name for display
        display_name = config_name.upper() if config_name in ['baseline', 'haxe'] else f'λ={config_name}'
        
        axes[idx].bar(token_indices, attention, alpha=0.7, color=color, label=f'{display_name} Attention')
        axes[idx].set_title(f'{model_key.upper()} {display_name} - Attention Distribution', 
                           fontsize=12, fontweight='bold', pad=10)
        axes[idx].set_ylabel('Attention Score', fontsize=10)
        
        metrics_text = f'Entropy: {entropy:.3f}'
        axes[idx].text(0.98, 0.95, metrics_text, transform=axes[idx].transAxes, 
                      fontsize=11, verticalalignment='top', horizontalalignment='right',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8, edgecolor='black', linewidth=1.5))
        axes[idx].legend(loc='upper left')
        axes[idx].grid(axis='y', alpha=0.3)
    
    # Ground Truth
    axes[-1].bar(token_indices, gt_mask, alpha=0.7, color='#DC143C', label='Ground Truth (Human Rationale)')
    axes[-1].set_title('Ground Truth Human Rationale', fontsize=12, fontweight='bold', pad=10)
    axes[-1].set_xlabel('Token Index', fontsize=10)
    axes[-1].set_ylabel('Rationale Mask', fontsize=10)
    axes[-1].legend(loc='upper left')
    axes[-1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed metrics
    print(f"\n{'='*80}")
    print(f"Example: {annotation_id} ({model_key.upper()})")
    print(f"{'='*80}")
    if doc_text and len(doc_text) < 500:
        print(f"Text: {doc_text}\n")
    print(f"Classification: {config_data[configs[0]].get('classification', 'N/A')}")
    
    # Get baseline entropy for comparison
    baseline_entropy = entropy_data.get('baseline', entropy_data.get('0', list(entropy_data.values())[0]))
    
    print(f"\n{'Configuration Metrics:':-<80}")
    for config_name in configs:
        entropy = entropy_data[config_name]
        if config_name == 'baseline' or entropy == baseline_entropy:
            print(f"  {config_name:>10}: Entropy = {entropy:.4f}")
        else:
            change_pct = (entropy - baseline_entropy) / baseline_entropy * 100
            print(f"  {config_name:>10}: Entropy = {entropy:.4f} ({change_pct:+.1f}%)")
    
    print(f"\n{'Entropy Progression Analysis:':-<80}")
    sorted_configs = sorted(configs, key=lambda x: entropy_data[x])
    print(f"  Most focused (lowest entropy): {sorted_configs[0]} = {entropy_data[sorted_configs[0]]:.4f}")
    print(f"  Least focused (highest entropy): {sorted_configs[-1]} = {entropy_data[sorted_configs[-1]]:.4f}")
    
    if baseline_entropy != entropy_data[sorted_configs[0]]:
        improvement = (baseline_entropy - entropy_data[sorted_configs[0]]) / baseline_entropy * 100
        print(f"  Best improvement vs baseline: {improvement:.2f}% {'✓ More focused' if improvement > 0 else '✗ Less focused'}")

In [64]:
# Find instances where BERT has much lower entropy than DistilBERT and DeBERTa
print("=" * 80)
print("SEARCHING FOR INSTANCES WITH BERT HAVING LOWER ENTROPY")
print("=" * 80)

# Check which models are available
available_models = sorted(model_data.keys())
print(f"\nAvailable models: {available_models}")

# Map model names
bert_key = None
distilbert_key = None
deberta_key = None

for model in available_models:
    if 'bert' in model.lower() and 'distil' not in model.lower() and 'deberta' not in model.lower():
        bert_key = model
    elif 'distil' in model.lower():
        distilbert_key = model
    elif 'deberta' in model.lower() or 'microsoft' in model.lower():
        deberta_key = model

print(f"\nMapped models:")
print(f"  BERT: {bert_key}")
print(f"  DistilBERT: {distilbert_key}")
print(f"  DeBERTa: {deberta_key}")

if not all([bert_key, distilbert_key, deberta_key]):
    print("\n⚠ Not all models found! Cannot perform comparison.")
else:
    # Get a common configuration (e.g., baseline or haxe)
    config_to_compare = None
    for config in ['haxe']:
        if (config in model_data.get(bert_key, {}) and 
            config in model_data.get(distilbert_key, {}) and 
            config in model_data.get(deberta_key, {})):
            config_to_compare = config
            break
    
    if not config_to_compare:
        print("\n⚠ No common configuration found across all models!")
    else:
        print(f"\nComparing configuration: {config_to_compare}")
        print("-" * 80)
        
        # Build entropy map for each annotation_id
        entropy_comparison = []
        
        # Get BERT data
        bert_data = {item['annotation_id']: item for item in model_data[bert_key][config_to_compare]}
        distilbert_data = {item['annotation_id']: item for item in model_data[distilbert_key][config_to_compare]}
        deberta_data = {item['annotation_id']: item for item in model_data[deberta_key][config_to_compare]}
        
        # Find common annotation IDs
        common_ids = set(bert_data.keys()) & set(distilbert_data.keys()) & set(deberta_data.keys())
        print(f"\nFound {len(common_ids)} common annotation IDs across all models")
        
        # Calculate entropies for common IDs
        for ann_id in common_ids:
            bert_attention = bert_data[ann_id]['rationales'][0]['soft_rationale_predictions']
            distilbert_attention = distilbert_data[ann_id]['rationales'][0]['soft_rationale_predictions']
            deberta_attention = deberta_data[ann_id]['rationales'][0]['soft_rationale_predictions']
            
            bert_entropy = calculate_attention_entropy(bert_attention)
            distilbert_entropy = calculate_attention_entropy(distilbert_attention)
            deberta_entropy = calculate_attention_entropy(deberta_attention)
            
            # Calculate how much lower BERT is
            diff_distilbert = distilbert_entropy - bert_entropy
            diff_deberta = deberta_entropy - bert_entropy
            avg_diff = (diff_distilbert + diff_deberta) / 2
            
            entropy_comparison.append({
                'annotation_id': ann_id,
                'bert_entropy': bert_entropy,
                'distilbert_entropy': distilbert_entropy,
                'deberta_entropy': deberta_entropy,
                'diff_vs_distilbert': diff_distilbert,
                'diff_vs_deberta': diff_deberta,
                'avg_diff': avg_diff,
                'diff_pct_distilbert': (diff_distilbert / distilbert_entropy) * 100,
                'diff_pct_deberta': (diff_deberta / deberta_entropy) * 100
            })
        
        # Sort by average difference (highest first)
        entropy_comparison.sort(key=lambda x: x['avg_diff'], reverse=True)
        
        # Display top 10 cases where BERT has much lower entropy
        print("\n" + "=" * 120)
        print("TOP 10 INSTANCES WHERE BERT HAS LOWER ENTROPY")
        print("=" * 120)
        print(f"{'Rank':<6} {'Annotation ID':<25} {'BERT':<10} {'DistilBERT':<12} {'DeBERTa':<10} {'Δ vs Dist':<12} {'Δ vs DeB':<12}")
        print("-" * 120)
        
        top_instances = []
        for i, item in enumerate(entropy_comparison[:10], 1):
            print(f"#{i:<5} {item['annotation_id']:<25} {item['bert_entropy']:<10.4f} "
                  f"{item['distilbert_entropy']:<12.4f} {item['deberta_entropy']:<10.4f} "
                  f"{item['diff_vs_distilbert']:<12.4f} {item['diff_vs_deberta']:<12.4f}")
            top_instances.append(item['annotation_id'])
        
        print("\n" + "=" * 120)
        print("BEST INSTANCE FOR VISUALIZATION")
        print("=" * 120)
        best_instance = entropy_comparison[35]
        print(f"Annotation ID: {best_instance['annotation_id']}")
        print(f"  BERT Entropy:       {best_instance['bert_entropy']:.4f}")
        print(f"  DistilBERT Entropy: {best_instance['distilbert_entropy']:.4f} "
              f"({best_instance['diff_pct_distilbert']:+.1f}% higher)")
        print(f"  DeBERTa Entropy:    {best_instance['deberta_entropy']:.4f} "
              f"({best_instance['diff_pct_deberta']:+.1f}% higher)")
        print(f"\n  BERT is more focused by {best_instance['avg_diff']:.4f} entropy units on average!")
        
        # Store for later use
        best_bert_instance = best_instance['annotation_id']

SEARCHING FOR INSTANCES WITH BERT HAVING LOWER ENTROPY

Available models: ['bert']

Mapped models:
  BERT: bert
  DistilBERT: None
  DeBERTa: None

⚠ Not all models found! Cannot perform comparison.


In [65]:
# Visualize the best instance across BERT, DistilBERT, and DeBERTa
if 'best_bert_instance' in locals():
    print("=" * 80)
    print(f"VISUALIZING BEST INSTANCE: {best_bert_instance}")
    print("=" * 80)
    
    # Reduced height, bigger fonts for paper
    fig, axes = plt.subplots(4, 1, figsize=(16, 11))
    
    models_to_plot = [
        (bert_key, 'BERT', '#3498db'),
        (distilbert_key, 'DistilBERT', '#e74c3c'),
        (deberta_key, 'DeBERTa', '#2ecc71')
    ]
    
    attention_values = {}
    entropy_values = {}
    
    for idx, (model_key, model_name, color) in enumerate(models_to_plot):
        # Find the data
        data_item = None
        for item in model_data[model_key][config_to_compare]:
            if item['annotation_id'] == best_bert_instance:
                data_item = item
                break
        
        if data_item:
            attention = data_item['rationales'][0]['soft_rationale_predictions']
            entropy = calculate_attention_entropy(attention)
            token_indices = np.arange(len(attention))
            
            attention_values[model_name] = attention
            entropy_values[model_name] = entropy
            
            axes[idx].bar(token_indices, attention, alpha=0.75, color=color, 
                         label=f'{model_name} Attention', edgecolor='black', linewidth=0.5)
            axes[idx].set_title(f'{model_name} - Attention Distribution', 
                               fontsize=14, fontweight='bold', pad=12)
            axes[idx].set_ylabel('Attention Score', fontsize=13, fontweight='bold')
            
            metrics_text = f'Entropy: {entropy:.3f}'
            axes[idx].text(0.98, 0.95, metrics_text, transform=axes[idx].transAxes, 
                          fontsize=13, verticalalignment='top', horizontalalignment='right', fontweight='bold',
                          bbox=dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.9, 
                                   edgecolor=color, linewidth=2.5))
            axes[idx].legend(loc='upper left', fontsize=12, framealpha=0.95)
            axes[idx].grid(axis='y', alpha=0.35, linestyle='--')
            axes[idx].tick_params(axis='both', which='major', labelsize=11)
            axes[idx].set_ylim(0, 0.4)
    
    # Ground Truth
    evidences = gt_model.get(bert_key, {}).get(best_bert_instance, {}).get('evidences', [])
    if not evidences:
        # Try other model keys
        for mk in [distilbert_key, deberta_key]:
            evidences = gt_model.get(mk, {}).get(best_bert_instance, {}).get('evidences', [])
            if evidences:
                break
    
    gt_mask = create_ground_truth_mask(evidences, len(attention))
    
    axes[3].bar(token_indices, gt_mask, alpha=0.8, color='#DC143C', 
               label='Ground Truth (Human Rationale)', edgecolor='darkred', linewidth=1)
    axes[3].set_title('Ground Truth Human Rationale', fontsize=14, fontweight='bold', pad=12)
    axes[3].set_xlabel('Token Index', fontsize=13, fontweight='bold')
    axes[3].set_ylabel('Rationale Mask', fontsize=13, fontweight='bold')
    axes[3].legend(loc='upper left', fontsize=12, framealpha=0.95)
    axes[3].grid(axis='y', alpha=0.35, linestyle='--')
    axes[3].tick_params(axis='both', which='major', labelsize=11)
    axes[3].set_ylim(0, 1)
    
    plt.tight_layout()
    
    # Export for printing
    output_filename = f'entropy_comparison_{best_bert_instance}.png'
    plt.savefig(output_filename, dpi=330, bbox_inches='tight', facecolor='white', edgecolor='none')
    print(f"\n✓ Figure saved as: {output_filename} (330 DPI)")
    
    # Also save as PDF for vector graphics (ideal for printing)
    output_pdf = f'entropy_comparison_{best_bert_instance}.pdf'
    plt.savefig(output_pdf, bbox_inches='tight', facecolor='white', edgecolor='none')
    print(f"✓ Figure saved as: {output_pdf} (vector format)")
    
    plt.show()
    
    print(f"\n{'='*80}")
    print(f"ENTROPY COMPARISON SUMMARY")
    print(f"{'='*80}")
    print(f"Annotation ID: {best_bert_instance}")
    print(f"\n{'Model':<15} {'Entropy':<12} {'vs BERT':<15} {'Focus Quality':<20}")
    print("-" * 80)
    models = ['BERT', 'DistilBERT', 'DeBERTa']
    for i, model in enumerate(models):
        entropy = entropy_values[model]
        if i == 0:
            print(f"{model:<15} {entropy:<12.4f} {'—':<15} {'⭐⭐⭐ MOST FOCUSED':<20}")
        else:
            diff_pct = ((entropy - entropy_values['BERT']) / entropy_values['BERT']) * 100
            print(f"{model:<15} {entropy:<12.4f} {f'+{diff_pct:.1f}%':<15} {'Less focused':<20}")
    
    print(f"\n💡 BERT's entropy is {((entropy_values['DistilBERT'] - entropy_values['BERT']) / entropy_values['BERT'] * 100):.1f}% lower than DistilBERT")
    print(f"💡 BERT's entropy is {((entropy_values['DeBERTa'] - entropy_values['BERT']) / entropy_values['BERT'] * 100):.1f}% lower than DeBERTa")
else:
    print("⚠ Run the previous cell first to identify the best instance!")

⚠ Run the previous cell first to identify the best instance!


In [66]:
# Find examples with ground truth for visualization
print("=" * 80)
print("FINDING EXAMPLES WITH GROUND TRUTH")
print("=" * 80)
model_key = 'distilbert'
print(f"\n{model_key.upper()} Examples:")
print("-" * 80)

examples_with_gt = []

# Get first config to iterate through samples
first_config = sorted([k for k in model_data[model_key].keys() if not k.endswith('_entropies')])[0]

for item in model_data[model_key][first_config][:100]:
    annotation_id = item['annotation_id']
    if annotation_id in gt_model.get(model_key, {}):
        evidences = gt_model[model_key][annotation_id]['evidences']
        if evidences and evidences[0]:  # Has non-empty evidences
            examples_with_gt.append(annotation_id)
            if len(examples_with_gt) >= 3:
                break

print(f"  Found {len(examples_with_gt)} examples with ground truth")
print(f"  Example IDs: {examples_with_gt[:3]}")

# Visualize first example
if examples_with_gt:
    print(f"\n  Visualizing example: {examples_with_gt[1]}")
    visualize_attention_example(examples_with_gt[0], model_key=model_key)

FINDING EXAMPLES WITH GROUND TRUTH

DISTILBERT Examples:
--------------------------------------------------------------------------------


KeyError: 'distilbert'

In [ ]:
# Test entropy calculation with dummy data
probs_dummy = np.array([0.1, 0.1, 0.1, 0.1, 0.1])
print(f"Uniform distribution entropy: {calculate_attention_entropy(probs_dummy):.4f}")

# More focused distribution
probs_focused = np.array([0.7, 0.1, 0.1, 0.05, 0.05])
print(f"Focused distribution entropy: {calculate_attention_entropy(probs_focused):.4f}")

Uniform distribution entropy: 1.1513
Focused distribution entropy: 1.0098


In [67]:
# Summary statistics across all models and configurations
print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

for model_key in sorted(model_data.keys()):
    print(f"\n{model_key.upper()} Configuration Summary:")
    print("-" * 80)
    
    configs = sorted([k for k in model_data[model_key].keys() if not k.endswith('_entropies')])
    
    for config_name in configs:
        entropies = model_data[model_key][f'{config_name}_entropies']
        print(f"  {config_name:>10}: n={len(entropies)}, "
              f"mean={np.mean(entropies):.4f}, "
              f"median={np.median(entropies):.4f}, "
              f"std={np.std(entropies):.4f}, "
              f"min={np.min(entropies):.4f}, "
              f"max={np.max(entropies):.4f}")

SUMMARY STATISTICS

BERT Configuration Summary:
--------------------------------------------------------------------------------
    baseline: n=1142, mean=2.9775, median=3.0021, std=0.5434, min=1.3406, max=4.0814
        haxe: n=1142, mean=2.3158, median=2.3165, std=0.4877, min=0.9604, max=3.7257
      lam0.1: n=1142, mean=2.6102, median=2.6168, std=0.5094, min=1.2895, max=3.8009
      lam0.5: n=1142, mean=2.2727, median=2.2638, std=0.4848, min=0.9317, max=3.7452
      lam1.0: n=1142, mean=2.2459, median=2.2391, std=0.4894, min=0.7983, max=3.6784
  lam100.0_m100.0: n=1142, mean=2.8670, median=2.9042, std=0.4976, min=1.3467, max=4.0250
      lam2.0: n=1142, mean=2.1963, median=2.2011, std=0.4709, min=0.9102, max=3.7130
       m0.05: n=1142, mean=2.6906, median=2.7192, std=0.5146, min=1.0763, max=3.9090
        m0.2: n=1142, mean=2.0180, median=2.0006, std=0.4947, min=0.8238, max=3.5060
        m0.3: n=1142, mean=1.7794, median=1.7327, std=0.5217, min=0.5250, max=3.4713
        m0.5: n=